# Model summaries

Prints the layers of each CoNNGaFit network, with each layer's output shape and parameter count, using
`conngafit.networks.model_summary` (a wrapper around `torchinfo.summary`). Change the settings in a section to see how
the architecture and its size respond, e.g. how the `"fc"` head's parameter count grows with the map size while the
conv heads' doesn't.

In [1]:
from conngafit.networks import model_summary

INPUT_SHAPE = (64, 64, 64)  # (nSpec, nX, nY) of one cube
DEPTH = 2  # Levels of nested layers to list; 3 or more lists the layers inside each block

## 1. Residual U-Net (`unet3d_resblock`)

Built once for each output head in `heads`. The fc-head settings (`n_fc`, `n_fc_layers`, `fc_pool_shape`) are unused by
the conv heads.

In [2]:
from conngafit.models.unet3d_resblock import NeuralNetwork as ResBlockNet

resblock_settings = dict(n_filt0=8, kernel_stem=(3, 3, 3), kernel_coder=(3, 3, 3), n_fc=64, n_stages=4)
heads = ["fc", "conv", "conv_halfres"]

for head in heads:
    model = ResBlockNet(INPUT_SHAPE, **resblock_settings, head=head)
    print(f"head={head!r}")
    print(model_summary(model, INPUT_SHAPE, depth=DEPTH))

head='fc'
Layer (type:depth-idx)                             Output Shape              Param #
NeuralNetwork                                      [1, 1, 4096]              --
├─Stem: 1-1                                        [1, 8, 16, 16, 16]        --
│    └─Conv3d: 2-1                                 [1, 8, 32, 32, 32]        224
│    └─MaxPool3d: 2-2                              [1, 8, 16, 16, 16]        --
│    └─LeakyReLU: 2-3                              [1, 8, 16, 16, 16]        --
├─ModuleList: 1-8                                  --                        (recursive)
│    └─Sequential: 2-4                             [1, 8, 16, 16, 16]        7,008
├─Sequential: 1-3                                  [1, 8, 16, 16, 16]        --
├─ModuleList: 1-8                                  --                        (recursive)
│    └─Sequential: 2-5                             [1, 16, 8, 8, 8]          24,560
├─Sequential: 1-5                                  [1, 16, 8, 8, 8]          --

## 2. Dilated residual network (`unet3d_dilated_resnet`)

Keeps every axis at full resolution (or downsamples once, with `stem_stride` > 1) and grows its receptive field with
dilated convolutions instead.

In [3]:
from conngafit.models.unet3d_dilated_resnet import NeuralNetwork as DilatedNet

dilated_settings = dict(n_filt0=8, kernel_stem=(3, 3, 3), kernel_coder=(3, 3, 3), n_fc=64, n_blocks=2)

for head in ["fc", "conv"]:
    model = DilatedNet(INPUT_SHAPE, **dilated_settings, head=head)
    print(f"head={head!r}")
    print(model_summary(model, INPUT_SHAPE, depth=DEPTH))

head='fc'
Layer (type:depth-idx)                   Output Shape              Param #
NeuralNetwork                            [1, 1, 4096]              --
├─Encoder: 1-1                           [1, 8, 64, 64, 64]        --
│    └─Conv3d: 2-1                       [1, 8, 64, 64, 64]        224
│    └─Identity: 2-2                     [1, 8, 64, 64, 64]        --
├─ModuleList: 1-2                        --                        --
│    └─DilatedResidualBlock: 2-3         [1, 8, 64, 64, 64]        12,264
│    └─DilatedResidualBlock: 2-4         [1, 8, 64, 64, 64]        12,264
├─Decoder: 1-3                           [1, 8, 64, 64, 64]        --
│    └─Conv3d: 2-5                       [1, 8, 64, 64, 64]        1,736
│    └─Identity: 2-6                     [1, 8, 64, 64, 64]        --
├─AdaptiveAvgPool3d: 1-4                 [1, 8, 8, 8, 8]           --
├─Sequential: 1-5                        [1, 64]                   --
│    └─Linear: 2-7                       [1, 64]               

## 3. Conditional diffusion model (`unet3d_ddpm`)

The table is of the denoiser, the network inside the diffusion process: it takes the cube, the noisy maps and the
timestep, and predicts the noise in the maps.

In [4]:
from conngafit.models.unet3d_ddpm import DenoisingUNet3d, GaussianDiffusion

denoiser = DenoisingUNet3d(INPUT_SHAPE, n_filt0=8, kernel_stem=(3, 3, 3), kernel_coder=(3, 3, 3), n_maps=1)
diffusion = GaussianDiffusion(denoiser, n_timesteps=100)
print(model_summary(diffusion, INPUT_SHAPE, depth=DEPTH))

Layer (type:depth-idx)                             Output Shape              Param #
DenoisingUNet3d                                    [1, 1, 64, 64]            --
├─TimestepEmbedding: 1-1                           [1, 32]                   --
│    └─Sequential: 2-1                             [1, 32]                   1,344
├─ConditionalStem: 1-2                             [1, 8, 16, 16, 16]        --
│    └─Conv3d: 2-2                                 [1, 8, 32, 32, 32]        440
│    └─MaxPool3d: 2-3                              [1, 8, 16, 16, 16]        --
│    └─LeakyReLU: 2-4                              [1, 8, 16, 16, 16]        --
├─ModuleList: 1-3                                  --                        --
│    └─ModuleList: 2-5                             --                        8,064
│    └─ModuleList: 2-6                             --                        26,672
│    └─ModuleList: 2-7                             --                        101,984
│    └─ModuleList: 